In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# SHAP Explainability

In [ ]:
!pip install shap -q
import shap
import numpy as np
import matplotlib.pyplot as plt
import joblib

In [ ]:
# Load feature matrices and splits
X_train_final, X_test_final = joblib.load('/kaggle/working/feature_matrices.pkl')
_, _, y_train, y_test = joblib.load('/kaggle/working/data_splits.pkl')

# Load XGBoost model
xgb = joblib.load('/kaggle/working/model_xgb.pkl')

print("Data and model loaded.")

In [ ]:
# Take a random sample of 100 test instances (SHAP can be slow on full set)
np.random.seed(42)
idx_sample = np.random.choice(X_test_final.shape[0], size=100, replace=False)
X_sample = X_test_final[idx_sample]
y_sample = y_test.iloc[idx_sample].values

print(f"Sampled {X_sample.shape[0]} instances.")

In [ ]:
# Create explainer (this may take 10-20 seconds on 100 samples)
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_sample)

print("SHAP values computed.")

In [ ]:
import matplotlib.pyplot as plt
import shap

# Force a small figure size before plotting
plt.figure(figsize=(4, 3))
shap.summary_plot(shap_values, X_sample, plot_type="bar", show=False, max_display=10)
plt.title("Mean |SHAP| Feature Importance", fontsize=10)
plt.tight_layout()
plt.savefig('/kaggle/working/shap_bar_highres.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: shap_bar_highres.png")

**WATER PLOT**

In [ ]:
# Explain a single prediction (waterfall plot) for one article

# Pick one correct and one misclassified example from the sample
import pandas as pd
from sklearn.metrics import accuracy_score

# Get predictions for the sample
preds_sample = xgb.predict(X_sample)
acc_sample = accuracy_score(y_sample, preds_sample)
print(f"Sample accuracy: {acc_sample:.4f}")

# Find indices of correct predictions and one misclassified
correct_idx = np.where(preds_sample == y_sample)[0]
mis_idx = np.where(preds_sample != y_sample)[0]

if len(mis_idx) > 0:
    # Waterfall for a misclassified example
    idx = mis_idx[0]
    shap.waterfall_plot(shap.Explanation(values=shap_values[idx],
                                         base_values=explainer.expected_value,
                                         data=X_sample[idx].toarray().flatten(),
                                         feature_names=[f"f{i}" for i in range(X_sample.shape[1])]),
                        show=False)
    plt.title(f"Waterfall plot (Misclassified: True={y_sample[idx]}, Pred={preds_sample[idx]})")
    plt.tight_layout()
    plt.savefig('/kaggle/working/shap_waterfall_mis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: shap_waterfall_mis.png")
else:
    # If no misclassifications, show a correct example
    idx = correct_idx[0]
    shap.waterfall_plot(shap.Explanation(values=shap_values[idx],
                                         base_values=explainer.expected_value,
                                         data=X_sample[idx].toarray().flatten(),
                                         feature_names=[f"f{i}" for i in range(X_sample.shape[1])]),
                        show=False)
    plt.title(f"Waterfall plot (Correctly classified: True={y_sample[idx]})")
    plt.tight_layout()
    plt.savefig('/kaggle/working/shap_waterfall_correct.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: shap_waterfall_correct.png")

In [ ]:
# Force plot for the first test instance
shap.force_plot(explainer.expected_value, shap_values[0,:], X_sample[0].toarray(),
                matplotlib=True, show=False)
plt.title("Force plot (single prediction)")
plt.tight_layout()
plt.savefig('/kaggle/working/shap_force.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_force.png")